In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
# import pandas_ta as ta
import gym_trading_env
from sb3_contrib import RecurrentPPO
import wandb
from gym_trading_env.wrapper import DiscreteActionsWrapper
from wandb.integration.sb3 import WandbCallback
from stable_baselines3 import DQN
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from stable_baselines3.common.monitor import Monitor 

# Import de nos propres fichiers
import reward as reward_functions
import features as features


In [2]:
from typing import Callable

def linear_schedule(initial_value: float) -> Callable[[float], float]:
    def func(progress_remaining: float) -> float:
        return progress_remaining * initial_value
    return func

def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

In [3]:
os.environ["WANDB_ERROR_REPORTING"] = "False" 
os.environ["WANDB_CONSOLE"] = "off"

# Model 1

On essaye un model basique, avec preprocess basic

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_basic,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model1 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model1.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model1.save("DQN_1")


wandb: Currently logged in as: marianne-billet (marianne-billet-cpe-lyon) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/qr4asx5b/DQN_1


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.name 
    
    while not done:
        action, _states = model1.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    if done:
        history = env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

# Model 2

On essaye avec preprocess dqn

In [6]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_dqn,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model2 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model2.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model2.save("DQN_2")


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/qbu21zu2/DQN_1
Market Return : 38.69%   |   Portfolio Return : -98.37%   |   
Market Return : 22.46%   |   Portfolio Return : -99.68%   |   
Market Return : 46.62%   |   Portfolio Return : -98.21%   |   


KeyboardInterrupt: 

In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.name 
    
    while not done:
        action, _states = model2.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    if done:
        history = env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

# Model 3

On essaye avec preprocess finance

In [3]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model3 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model3.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model3.save("DQN_3")


wandb: Currently logged in as: marianne-billet (marianne-billet-cpe-lyon) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/flr4r9e7/DQN_1


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7457c8c9cef0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7457c225af30, execution_count=3 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7457c8ee2ba0, raw_cell="env = gym.make(
    "MultiDatasetTradingEnv",
    .." transformed_cell="env = gym.make(
    "MultiDatasetTradingEnv",
    .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/cpe/Bureau/RL/reinforcement-learning-project/DQN.ipynb#X14sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.name 
    
    while not done:
        action, _states = model3.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    if done:
        history = env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

# Model 4

On essaye avec feature preprocess finance finale

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance_finale,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env = DiscreteActionsWrapper(env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model4 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model4.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model4.save("DQN_4")


wandb: Currently logged in as: marianne-billet (marianne-billet-cpe-lyon) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/vnxq7nwh/DQN_1


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.name 
    
    while not done:
        action, _states = model4.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    if done:
        history = env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

# Model 5

On essaye avec une fonction de récompense en log

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance_finale,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_functions.reward_log_returns
)

env = DiscreteActionsWrapper(env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model5 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model5.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model5.save("DQN_5")


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/2m2bun6x/DQN_1
Market Return : 26.64%   |   Portfolio Return : -53.71%   |   


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.name 
    
    while not done:
        action, _states = model5.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    if done:
        history = env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

# Model 6

On essaye avec une fonction de récompense rapport market

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=features.preprocess_finance_finale,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_functions.reward_log_returns
)

env = DiscreteActionsWrapper(env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model6 = DQN(
    "MlpPolicy", 
    env, 
    verbose=1,
    tensorboard_log=f"runs/{run.id}",
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

model6.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model6.save("DQN_6")


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/lyjaqxau/DQN_1
Market Return : 42.46%   |   Portfolio Return : -62.96%   |   


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.name 
    
    while not done:
        action, _states = model6.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    if done:
        history = env.historical_info
        df_history = pd.DataFrame(history.history_storage, columns=history.columns)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()